# Lab 10: Multimodal Live Interactions (Gemini 3.1 Flash Live)

The **Multimodal Live API** is a low-latency WebSocket service. Gemini 3.1 Flash Live is strict: it requires manual tool handling and specific response modalities.

## Objectives
1. Perform a simple **Live Handshake** using the AUDIO modality.
2. Handle **Live Transcriptions** for real-time text output.
3. Implement **Manual Tool Response Handling** using top-level message checks.

In [ ]:
import os

from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Client initialized.")

## 1. Basic Live Chat

Verifying the connection with a simple text-in, transcription-out flow.

In [ ]:
async def run_simple_live():
    model_id = "gemini-3.1-flash-live-preview"
    config = types.LiveConnectConfig(response_modalities=["AUDIO"])

    try:
        async with client.aio.live.connect(model=model_id, config=config) as session:
            print(f"Connected to {model_id}")
            await session.send_realtime_input(
                text="Say 'Connection successful' if you hear me."
            )

            print("Gemini: ", end="")
            async for message in session.receive():
                if (message.server_content
                    and message.server_content.output_transcription
                ):
                    print(
                        message.server_content.output_transcription.text,
                        end="",
                        flush=True
                    )

                if message.server_content and message.server_content.turn_complete:
                    print("\n--- Turn Complete ---")
                    break
    except Exception as e:
        print(f"\nSimple Live Error: {e}")

await run_simple_live()

## 2. Simplified Tool Handling

As per the official documentation, tool calls appear as top-level fields in the response. We must handle them manually and send back a `LiveClientToolResponse`.

In [ ]:
async def run_live_with_tools():
    model_id = "gemini-3.1-flash-live-preview"

    # Define tool schema using SDK structured types
    get_temp_tool = types.Tool(
        function_declarations=[
            types.FunctionDeclaration(
                name="get_room_temperature",
                description="Returns the current room temperature.",
                parameters=types.Schema(type="OBJECT", properties={})
            )
        ]
    )

    config = types.LiveConnectConfig(
        tools=[get_temp_tool],
        response_modalities=["AUDIO"]
    )

    async with client.aio.live.connect(model=model_id, config=config) as session:
        print("--- Connected with Tools ---")
        await session.send_realtime_input(text="What is the temperature in this room?")

        print("Gemini: ", end="")
        async for message in session.receive():
            # 1. Check for Tool Calls (Direct top-level check)
            if message.tool_call:
                for call in message.tool_call.function_calls:
                    print(f"\n[Model Request] Tool Call: {call.name}")

                    # Send manual response back
                    tool_resp = types.FunctionResponse(
                        name=call.name,
                        id=call.id,
                        response={"result": "22.5 degrees Celsius"}
                    )

                    await session.send_tool_response(function_responses=tool_resp)

            # 2. Check for Text Output (Transcription)
            if message.server_content and message.server_content.output_transcription:
                print(
                    message.server_content.output_transcription.text,
                    end="",
                    flush=True
                )

            # 3. End session on turn completion
            if message.server_content and message.server_content.turn_complete:
                print("\n--- Interaction Complete ---")
                break

await run_live_with_tools()

## Summary

1. **AUDIO Modality**: Essential for stability in Gemini 3.1 Live.
2. **Top-level Checks**: Intercepting `message.tool_call` directly is the most reliable way to handle agents.
3. **Transcriptions**: Used `output_transcription.text` to get real-time feedback while the model streams audio.